<a href="https://colab.research.google.com/github/Donietta/RealTimeAudioProcessing/blob/main/S1-Python_AudioProcessor/audio_processor.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Audio Processor

This notebook reads a WAV audio file, applies a processing step, and saves the result.

## 1. Install dependencies

In [1]:
!pip install -q numpy scipy

## 2. Input audio file

Run the following cell to download the original Marimba.wav file.

In [2]:
!wget -q https://raw.githubusercontent.com/ThomasHezard/RealTimeAudioProcessing/main/S1-Python_AudioProcessor/Marimba.wav

Uncomment and run the following cell to upload your audio wav file.
Adjust the input file name in next cell if needed.

In [3]:
#from google.colab import files
#uploaded = files.upload()

## 3. Read input file

In [4]:
import numpy as np
import scipy.io.wavfile as wavfile


# Parameters
# **********

# This must be an audio file readable by scipy.io.wavfile.read() function
input_file = 'Marimba.wav'
output_file = 'output.wav'


# Read input file
# ***************

fs, input_data = wavfile.read(input_file, mmap=True)
# convert to float
max_value = float(-np.iinfo(input_data.dtype).min)
input_data = input_data.astype('float32') / max_value
# convert to mono if not already
if len(input_data.shape) > 1:
    input_data = np.mean(input_data, axis=1)
input_data = input_data.flatten()

print(f'Sample rate: {fs} Hz')
print(f'Duration: {len(input_data)/fs:.2f} s')
print(f'Samples: {len(input_data)}')

Sample rate: 44100 Hz
Duration: 5.18 s
Samples: 228287


## 4. Audio data process

Edit this cell to implement your audio processing.

In [13]:
output_data = np.zeros(len(input_data))

# *************************
# * PROCESSING COMES HERE *
# *************************

output_data = input_data


# *************************
# * DELAY TEST *
# *************************
output_data = input_data
f_e = 44100
def delay_function(d, input_data):  #ordre de grandeur de d (samples) -> ordre de grandeur f_e = 44100 Hz
  a = 1/2
  assert np.abs(a)<1
  for i in range (len(input_data)):
    if (i-d)<0 : #comme condition initiale on peut aussi mettre d à 0 si (i-d)<0
      output_data[i] = input_data[i]
    else :
      output_data[i]=input_data[i]+a*output_data[i-d]
  return output_data

output_data=delay_function(int(0.5*f_e), input_data)

#si faible delay --> les répetitions sont plus difficile à entendre
#si d=1 --> filtre AR d'ordre 1 --> filtre passe-bas
#on code un delay et on arrive à un filtre passe: plus le delay est faible plus on a de mal à entendre les oscillations à cause du pouvoir séparateur de l'oeille : t_séparation<20ms l'oreille n'entend pas deux sons distincts

# *************************
# * DISTORSION EXPONENTIELLE *
# *************************
def sign(q):
  if q<0 :
    return (-1)
  if q == 0 :
    return 0
  if q>0:
    return 1

def distorsion_function(input_data,gain):
    #input_data_with_gain = []
    input_data_with_gain = input_data
    output_distorsion = input_data
    for i in range (len(input_data)):
      input_data_with_gain[i] = input_data[i] * gain
      #input_data_with_gain.append(input_data_with_gain[i])
    for i in range (len(input_data)):
      output_distorsion[i] = sign(input_data_with_gain[i]) * (1-np.exp(-np.abs(input_data_with_gain[i])))
      #output_distorsion.append(output_distorsion[i])
    return output_distorsion

output_data_distorsion = distorsion_function(input_data, 500)
# *************************
# *************************

## 5. Save and download output

In [14]:
from google.colab import files

output_data = 0.99 * output_data / max(abs(output_data))
wavfile.write(output_file, int(fs), (output_data*np.iinfo(np.int16).max).astype(np.int16))

files.download(output_file)

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>